In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss


c:\Users\Zaineb\anaconda3\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [ ]:
# Load a pre-trained model for generating embeddings
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')  # A lightweight model; adjust as needed


In [ ]:
import pdfplumber

# Function to extract text from PDF and split it into chunks (pages, sections, or paragraphs)
def extract_text_from_pdf(pdf_path, chunk_size=1000):
    all_text = ""
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                all_text += page_text

    # Split text into chunks of a given size
    for i in range(0, len(all_text), chunk_size):
        chunks.append(all_text[i:i+chunk_size])
    
    return chunks

# Example usage:
pdf_file_path = './decrypted_output.pdf'
chunks = extract_text_from_pdf(pdf_file_path)
print(f"Total chunks extracted: {len(chunks)}")
    

In [ ]:
# Generate embeddings for each chunk of text
doc_embeddings = np.array([model.encode(chunk) for chunk in chunks])

# Create a FAISS index for fast similarity search
index = faiss.IndexFlatL2(doc_embeddings.shape[1])  # L2 similarity index
index.add(doc_embeddings)

print(f"Indexed {len(chunks)} document chunks.")


In [ ]:
# Function to search for the most relevant chunks
def search_relevant_chunks(query, k=3):
    query_embedding = model.encode(query)  # Encode the user's query
    D, I = index.search(np.array([query_embedding]), k)  # Search for the top-k most relevant chunks
    
    relevant_chunks = [chunks[idx] for idx in I[0]]  # Retrieve the relevant chunks
    return relevant_chunks

# Example user query
user_query = "What is the purpose of Quantitative Risk Analysis?"

# Perform the search
relevant_chunks = search_relevant_chunks(user_query, k=3)  # Get top 3 relevant chunks

# Display the most relevant chunks
for i, chunk in enumerate(relevant_chunks):
    print(f"Relevant Chunk {i+1}:\n{chunk[:500]}...\n")  # Print first 500 characters


In [ ]:
# Function to construct the chatbot prompt from the relevant chunk
def construct_prompt_from_chunk(chunk, user_query):
    prompt = f"""
    You are an expert assistant tasked with providing information from the document.
    The user has asked: "{user_query}"
    
    Here is the relevant document content:
    {chunk}
    
    Please provide an accurate and concise response based on this information.
    """
    return prompt

# Construct the prompt from the most relevant chunk
relevant_chunk = relevant_chunks[0]  # Take the first most relevant chunk
prompt = construct_prompt_from_chunk(relevant_chunk, user_query)
print(prompt)  # Check the prompt


In [ ]:
# Assuming you have a function to interact with your chatbot
def ask_chatbot(prompt):
    # Here you would call the actual chatbot API (e.g., Groq LLaMA)
    # This is a placeholder for demonstration
    response = "Chatbot response to the prompt"  # Replace with actual chatbot call
    return response

# Send the prompt to the chatbot and get the response
response = ask_chatbot(prompt)
print(f"Chatbot response: {response}")


In [ ]:
def chatbot_from_pdf(pdf_path, user_query):
    # Step 1: Extract text and split into chunks
    chunks = extract_text_from_pdf(pdf_path)
    
    # Step 2: Generate embeddings and index them in FAISS
    doc_embeddings = np.array([model.encode(chunk) for chunk in chunks])
    index = faiss.IndexFlatL2(doc_embeddings.shape[1])
    index.add(doc_embeddings)
    
    # Step 3: Search for the most relevant chunks
    relevant_chunks = search_relevant_chunks(user_query, k=3)
    
    # Step 4: Construct the chatbot prompt using the most relevant chunk
    relevant_chunk = relevant_chunks[0]  # Take the most relevant chunk
    prompt = construct_prompt_from_chunk(relevant_chunk, user_query)
    
    # Step 5: Pass the prompt to the chatbot
    response = ask_chatbot(prompt)
    
    return response

# Example usage:
user_query = "What is Quantitative Risk Analysis?"
response = chatbot_from_pdf(pdf_file_path, user_query)
print(f"Chatbot response: {response}")
